# imports

In [2]:
import mysql.connector
import pandas as pd
import numpy as np

from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Path relative to Scripts/
data_dir = Path("../Data")
input_file = data_dir / "bank_dataset_cleaned_2026-06-01.csv"

df = pd.read_csv(input_file, encoding="utf-8-sig")

print(f"Dataset loaded from: {input_file.resolve()}")
print(f"Shape: {df.shape}")

Dataset loaded from: C:\Users\nowan\Documents\itacademy\Simulador\ProjecteData\Equip_32\Data\bank_dataset_cleaned_2026-06-01.csv
Shape: (9519, 20)


# Data Transformations

# 1 Demografical clustering

In [4]:
# Age_group
bins   = [17, 25, 35, 50, 65, 100]
labels = [
    "Young (18-25)",
    "Young Adult (26-35)",
    "Adult (36-50)",
    "Middle-Aged (51-65)",
    "Senior (65+)"
]

df["age_group"] = pd.cut(
    df["age"],
    bins=bins,
    labels=labels,
    right=True    # right-closed intervals: (17,25] includes 25
)

In [5]:
counts = df["age_group"].value_counts().sort_index()
print(counts)
print(f"\nUnassigned (NaN): {df['age_group'].isna().sum()}")

age_group
Young (18-25)           410
Young Adult (26-35)    3359
Adult (36-50)          3617
Middle-Aged (51-65)    1759
Senior (65+)            374
Name: count, dtype: int64

Unassigned (NaN): 0


In [6]:
age_summary = (
    df.groupby("age_group", observed=True)["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

age_summary["n_clients"] = df.groupby("age_group", observed=True).size()
print(age_summary)

deposit              pct_no  pct_yes  n_clients
age_group                                      
Young (18-25)        0.2707   0.7293        410
Young Adult (26-35)  0.5138   0.4862       3359
Adult (36-50)        0.5809   0.4191       3617
Middle-Aged (51-65)  0.4997   0.5003       1759
Senior (65+)         0.1845   0.8155        374


In [7]:
# Financial Burden

# "unknown" is treated as 0 (no burden assumed)
# This is a deliberate modelling choice — document it in the notebook

burden_map = {"yes": 1, "no": 0, "unknown": 0}

df["default_score"]  = df["default"].map(burden_map)
df["housing_score"]  = df["housing"].map(burden_map)
df["loan_score"]     = df["loan"].map(burden_map)

In [8]:
df["financial_burden"] = (
    df["default_score"] +
    df["housing_score"] +
    df["loan_score"]
)

In [9]:
burden_labels = {
    0: "No burden",
    1: "Low burden",
    2: "Medium burden",
    3: "High burden"
}

df["financial_burden_label"] = df["financial_burden"].map(burden_labels)

In [10]:
burden_summary = (
    df.groupby("financial_burden_label")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

burden_summary["n_clients"] = df.groupby("financial_burden_label").size()

# Sort by score for readability
burden_summary = burden_summary.reindex(burden_labels.values())
print(burden_summary)

deposit                 pct_no  pct_yes  n_clients
financial_burden_label                            
No burden               0.3779   0.6221       4554
Low burden              0.6275   0.3725       4212
Medium burden           0.6944   0.3056        733
High burden             0.6500   0.3500         20


In [11]:
# education

print(df["education"].value_counts())
print(f"\nUnknown count: {(df['education'] == 'unknown').sum()}")


education
secondary    4647
tertiary     3188
primary      1259
unknown       425
Name: count, dtype: int64

Unknown count: 425


In [12]:
# Ordinal scale: unknown → NaN (excluded from ranking)
# primary=1, secondary=2, tertiary=3

education_order = {
    "primary"   : 1,
    "secondary" : 2,
    "tertiary"  : 3,
    "unknown"   : None
}

df["education_rank"] = df["education"].map(education_order)

In [13]:
education_labels = {
    "primary"   : "Primary",
    "secondary" : "Secondary",
    "tertiary"  : "Tertiary",
    "unknown"   : "Unknown"
}

df["education_label"] = df["education"].map(education_labels)

In [14]:
edu_summary = (
    df.groupby("education_label")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

edu_summary["n_clients"] = df.groupby("education_label").size()

# Sort by ordinal rank
order = ["Primary", "Secondary", "Tertiary", "Unknown"]
edu_summary = edu_summary.reindex(order)
print(edu_summary)

deposit          pct_no  pct_yes  n_clients
education_label                            
Primary          0.5965   0.4035       1259
Secondary        0.5425   0.4575       4647
Tertiary         0.4448   0.5552       3188
Unknown          0.4612   0.5388        425


In [15]:
#jobs

print(df["job"].value_counts())
print(f"\nUnknown count: {(df['job'] == 'unknown').sum()}")

job
management       2190
blue-collar      1618
technician       1541
admin.           1151
services          780
retired           699
self-employed     341
student           334
unemployed        307
entrepreneur      273
housemaid         226
unknown            59
Name: count, dtype: int64

Unknown count: 59


In [16]:
job_profile_map = {
    "admin."       : "White Collar",
    "management"   : "White Collar",
    "technician"   : "White Collar",
    "blue-collar"  : "Blue Collar",
    "housemaid"    : "Blue Collar",
    "services"     : "Blue Collar",
    "entrepreneur" : "Self-Employed",
    "self-employed": "Self-Employed",
    "retired"      : "Retired",
    "student"      : "Student",
    "unemployed"   : "Unemployed/Unknown",
    "unknown"      : "Unemployed/Unknown"
}

df["job_profile"] = df["job"].map(job_profile_map)

In [17]:
job_summary = (
    df.groupby("job_profile")["deposit"]
    .value_counts(normalize=True)
    .unstack()
    .rename(columns={1: "pct_yes", 0: "pct_no"})
    .round(4)
)

job_summary["n_clients"] = df.groupby("job_profile").size()

order = ["White Collar", "Blue Collar", "Self-Employed", "Unemployed/Unknown", "Retired", "Student"]
job_summary = job_summary.reindex(order)
print(job_summary)

deposit             pct_no  pct_yes  n_clients
job_profile                                   
White Collar        0.5018   0.4982       4882
Blue Collar         0.6220   0.3780       2624
Self-Employed       0.5749   0.4251        614
Unemployed/Unknown  0.4235   0.5765        366
Retired             0.3119   0.6881        699
Student             0.2335   0.7665        334


In [18]:
df


,id,age,job,marital,education,default,balance,housing,loan,contact,...,had_previous_contact,age_group,default_score,housing_score,loan_score,financial_burden,financial_burden_label,education_rank,education_label,job_profile
0,1,59.0,admin.,married,secondary,no,2343,yes,no,unknown,...,0,Middle-Aged (51-65),0,1,0,1,Low burden,2.0,Secondary,White Collar
1,4,55.0,services,married,secondary,no,2476,yes,no,unknown,...,0,Middle-Aged (51-65),0,1,0,1,Low burden,2.0,Secondary,Blue Collar
2,5,54.0,admin.,married,tertiary,no,184,no,no,unknown,...,0,Middle-Aged (51-65),0,0,0,0,No burden,3.0,Tertiary,White Collar
3,6,42.0,management,single,tertiary,no,0,yes,yes,unknown,...,0,Adult (36-50),0,1,1,2,Medium burden,3.0,Tertiary,White Collar
4,8,60.0,retired,divorced,secondary,no,545,yes,no,unknown,...,0,Middle-Aged (51-65),0,1,0,1,Low burden,2.0,Secondary,Retired
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9514,10638,33.0,technician,married,secondary,no,218,yes,yes,telephone,...,0,Young Adult (26-35),0,1,1,2,Medium burden,2.0,Secondary,White Collar
9515,10639,42.0,management,single,tertiary,no,1146,yes,no,unknown,...,0,Adult (36-50),0,1,0,1,Low burden,3.0,Tertiary,White Collar
9516,10640,31.0,unemployed,single,unknown,no,167,no,no,cellular,...,0,Young Adult (26-35),0,0,0,0,No burden,NaN,Unknown,Unemployed/Unknown
9517,10641,30.0,blue-collar,single,secondary,yes,447,no,no,cellular,...,1,Young Adult (26-35),1,0,0,1,Low burden,2.0,Secondary,Blue Collar


# CSV expoirt

In [19]:
# Original columns kept + new transformed columns
# Expand this list when financial & marketing transformations are added

cols_to_export = [
    # --- original ---
    "age", "job", "marital", "education",
    "default", "housing", "loan", "deposit",
    # --- transformed ---
    "age_group",
    "education_rank", "education_label",
    "job_profile",
    "financial_burden", "financial_burden_label",
    # --- pending (add when ready) ---
    # "campaign_group",
    # "duration_group",
    # "pdays_group"
]

df_export = df[cols_to_export].copy()
print(f"Shape: {df_export.shape}")
print(df_export.dtypes)

Shape: (9519, 14)
age                        float64
job                         object
marital                     object
education                   object
default                     object
housing                     object
loan                        object
deposit                      int64
age_group                 category
education_rank             float64
education_label             object
job_profile                 object
financial_burden             int64
financial_burden_label      object
dtype: object


In [20]:
# Quick sanity check — run before any export
print("=== Shape ===")
print(df_export.shape)

print("\n=== Nulls per column ===")
print(df_export.isnull().sum())

print("\n=== Sample ===")
print(df_export.head(3))

=== Shape ===
(9519, 14)

=== Nulls per column ===
age                         0
job                         0
marital                     0
education                   0
default                     0
housing                     0
loan                        0
deposit                     0
age_group                   0
education_rank            425
education_label             0
job_profile                 0
financial_burden            0
financial_burden_label      0
dtype: int64

=== Sample ===
    age       job  marital  education default housing loan  deposit  \
0  59.0    admin.  married  secondary      no     yes   no        1   
1  55.0  services  married  secondary      no     yes   no        1   
2  54.0    admin.  married   tertiary      no      no   no        1   

             age_group  education_rank education_label   job_profile  \
0  Middle-Aged (51-65)             2.0       Secondary  White Collar   
1  Middle-Aged (51-65)             2.0       Secondary   Blue Collar   

In [22]:
# Export transformed dataset
output_dir = Path("../Data")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "bank_dataset_transformed_2026-07-01.csv"

df_export.to_csv(
    output_file,              # ← fixed: was using plain filename instead of Path
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV saved to: {output_file.resolve()}")

CSV saved to: C:\Users\nowan\Documents\itacademy\Simulador\ProjecteData\Equip_32\Data\bank_dataset_transformed_2026-07-01.csv
